# 1. Test Transformer Modules

In [1]:
import torch
import os
import sys
sys.path.append('./src')

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

2.11.0+cu130
True
NVIDIA GeForce RTX 4080 SUPER
VRAM: 16.0 GB


In [2]:
from attention import MultiHeadAttention

d_model, num_heads, batch, seq_len = 512, 8, 2, 10

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch, seq_len, d_model)
output, attn_weights = mha(x, x, x)

print("Output shape:", output.shape)       # (2, 10, 512)
print("Attn weights:", attn_weights.shape) # (2, 8, 10, 10)

Output shape: torch.Size([2, 10, 512])
Attn weights: torch.Size([2, 8, 10, 10])


In [3]:
from feedforward import FeedForward

d_model, d_ff = 512, 2048
ff = FeedForward(d_model, d_ff)
x = torch.randn(2, 10, d_model)
output = ff(x)

print("Output shape:", output.shape)  # (2, 10, 512)

Output shape: torch.Size([2, 10, 512])


In [4]:
from layer_norm import LayerNorm

ln = LayerNorm(d_model)
x = torch.randn(2, 10, d_model)
output = ln(x)

print("Output shape:", output.shape)          # (2, 10, 512)
print("Mean (≈0):", output.mean(-1).mean().item())
print("Std  (≈1):", output.std(-1).mean().item())

Output shape: torch.Size([2, 10, 512])
Mean (≈0): -1.516309522386905e-09
Std  (≈1): 0.9999990463256836


In [5]:
from encoder import Encoder

encoder = Encoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)
x = torch.randn(2, 10, 512)
output = encoder(x)

print("Output shape:", output.shape)  # (2, 10, 512)

Output shape: torch.Size([2, 10, 512])


In [6]:
from decoder import Decoder

decoder = Decoder(d_model=512, num_heads=8, d_ff=2048, num_layers=6)

encoder_output = torch.randn(2, 10, 512)  # from encoder
tgt = torch.randn(2, 7, 512)             # target sequence

output = decoder(tgt, encoder_output)

print("Output shape:", output.shape)  # (2, 7, 512)

Output shape: torch.Size([2, 7, 512])


In [7]:
from transformer import Transformer

model = Transformer(src_vocab_size=10000, tgt_vocab_size=10000)
model = model.cuda()

# Dummy token indices
src = torch.randint(0, 10000, (2, 10)).cuda() 
tgt = torch.randint(0, 10000, (2, 7)).cuda() 

src_mask = model.make_src_mask(src, pad_idx = 0)
tgt_mask = model.make_tgt_mask(tgt, pad_idx = 0)

output = model(src, tgt, src_mask, tgt_mask)

print("Output shape:", output.shape)  # (2, 7, 10000)

Output shape: torch.Size([2, 7, 10000])


# 2. Test Tokenizer Module

In [11]:
from datasets import load_dataset

dataset = load_dataset("opus100", "en-ko")
print(dataset)
print(dataset['train'][0])

README.md: 0.00B [00:00, ?B/s]

en-ko/test-00000-of-00001.parquet:   0%|          | 0.00/143k [00:00<?, ?B/s]

en-ko/train-00000-of-00001.parquet:   0%|          | 0.00/70.1M [00:00<?, ?B/s]

en-ko/validation-00000-of-00001.parquet:   0%|          | 0.00/144k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 1000000
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
{'translation': {'en': "They're shaped like a bus.", 'ko': '할머니처럼 만들었지만.. ? 엉망이지만..'}}


In [13]:
os.makedirs('../data/raw', exist_ok=True)

# Save Korean and English sentences to text files for tokenizer training
with open('../data/raw/ko.txt', 'w', encoding='utf-8') as f:
    for item in dataset['train']:
        f.write(item['translation']['ko'] + '\n')

with open('../data/raw/en.txt', 'w', encoding='utf-8') as f:
    for item in dataset['train']:
        f.write(item['translation']['en'] + '\n')

print(f"Total sentences: {len(dataset['train'])}")
print("Sample KO:", dataset['train'][0]['translation']['ko'])
print("Sample EN:", dataset['train'][0]['translation']['en'])

Total sentences: 1000000
Sample KO: 할머니처럼 만들었지만.. ? 엉망이지만..
Sample EN: They're shaped like a bus.


In [15]:
from tokenizer import Tokenizer

sys.path.append('./src')
os.makedirs('../data/processed', exist_ok=True)

# Train Korean tokenizer
ko_tokenizer = Tokenizer()
ko_tokenizer.train(
    input_file='../data/raw/ko.txt',
    model_prefix='../data/processed/ko_tokenizer',
    vocab_size=16000
)

# Train English tokenizer
en_tokenizer = Tokenizer()
en_tokenizer.train(
    input_file='../data/raw/en.txt',
    model_prefix='../data/processed/en_tokenizer',
    vocab_size=16000
)

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: ../data/raw/ko.txt
  input_format: 
  model_prefix: ../data/processed/ko_tokenizer
  model_type: BPE
  vocab_size: 16000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 1
  bos_id: 2
  eos_id: 3
  pad_id: 0
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differe

KeyboardInterrupt: 

In [ ]:
# Test Korean tokenizer
text_ko = "나는 학교에 갑니다"
encoded = ko_tokenizer.encode(text_ko)
decoded = ko_tokenizer.decode(encoded)
print(f"Original : {text_ko}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")

# Test English tokenizer
text_en = "I go to school"
encoded = en_tokenizer.encode(text_en)
decoded = en_tokenizer.decode(encoded)
print(f"Original : {text_en}")
print(f"Encoded  : {encoded}")
print(f"Decoded  : {decoded}")